In [1]:
# Requisitar dados da API do Google Earth Engine
import ee
import geemap
import geopandas as gpd
import pandas as pd
import os
import matplotlib.pyplot as plt
import xarray as xr
import rioxarray as rxr

In [ ]:
# 1. Inicialização | ee.Authenticate()
ee.Initialize(
    project='ee-gabriel-495521',
    opt_url='https://earthengine-highvolume.googleapis.com',
)

# Set the path to the JSON file containing the geometry.
path_json = "../data/bacias_meso_SF.geojson"
gdf = gpd.read_file(path_json)
gdf_menor = gdf[gdf['nm_mesoRH'] == 'Baixo São Francisco']
geom_ee = geemap.geopandas_to_ee(gdf_menor)
area = geom_ee.geometry()

# 3. Carregar a coleção MODIS (Reflectância de Superfície)
xavier = ee.ImageCollection('MODIS/061/MOD09A1') \
    .filterBounds(area) \
    .filterDate('2000-02-18', '2025-12-31')

In [68]:
# 1.1 Função de Rescalonamento
def scaleBand(image):
    scaledImage = image.multiply(scale).add(offset)

    return scaledImage.copyProperties(image, image.propertyNames())

In [ ]:
# 3. Definir Parâmetros

# 3.1 Parâmetros de rescale para as variáveis
rescale_params = {
    "Rs": {"scale": 0.15708661, "offset": -0.057087},
    "u2": {"scale": 0.05905512, "offset": -0.059055},
    "Tmax": {"scale": 0.00106815, "offset": 15.0},
    "Tmin": {"scale": 0.00106815, "offset": 15.0},
    "RH": {"scale": 0.39370079, "offset": -0.393701},
    "pr": {"scale": 0.00686666, "offset": 225.0},
    "ET": {"scale": 0.05118110, "offset": 0.0},
}

# 3.2 Definir Variáveis de interesse
variaveis_interesse = ["Tmax", "pr", "RH", "ET", "Tmin"]

# 3.3 Definir Período de interesse
start = "1961-01-01"
end = "2024-03-20"

# 3.4 Definir Pasta de saída
output_folder = r"C:\Users\Gabriel\OneDrive - I Care\I Care - 11. GEO\1. Polo Clima\1. Projetos Ativos\MAU1\data\raw"

In [71]:
# Loop através de todas as variáveis de interesse
for var_name in variaveis_interesse:
    print(f"Processando variável: {var_name}")

    # Definir escala e offset para a variável atual
    scale = rescale_params[var_name]["scale"]
    offset = rescale_params[var_name]["offset"]

    # Carregar coleção filtrada
    col = (
        ee.ImageCollection("projects/ee-alexandrexavier/assets/BR-DWGD")
        .filterBounds(area)
        .filterDate(start, end)
        .map(scaleBand)
        .select(var_name)
    )

    # Converter para xarray
    try:
        ds = xr.open_dataset(
            col,
            engine="ee",
            geometry=area,
            scale=0.1,
            projection=col.first().select(0).projection(),
            fast_time_slicing=True,
            chunks={"index": 24},
        ).rename({"lon": "x", "lat": "y"})

        # Reorganizar dimensões e definir CRS
        ds = ds.transpose("time", "y", "x").rio.write_crs("EPSG:5880")

        # Configurar encoding para compressão
        encoding = {var_name: {"zlib": True}}

        # Caminho do arquivo de saída
        output_path = os.path.join(
            output_folder, f"{var_name}_Xavier_manaus_1961_2024.nc"
        )

        # Exportar para NetCDF
        ds.to_netcdf(
            output_path,
            encoding=encoding,
            format="NETCDF4",
        )

        print(f"Arquivo salvo com sucesso: {output_path}")

    except Exception as e:
        print(f"Erro ao processar {var_name}: {str(e)}")

print("Processamento concluído para todas as variáveis!")

Processando variável: Tmax
Arquivo salvo com sucesso: C:\Users\Gabriel\OneDrive - I Care\I Care - 11. GEO\1. Polo Clima\1. Projetos Ativos\MAU1\data\raw\Tmax_Xavier_manaus_1961_2024.nc
Processando variável: pr
Arquivo salvo com sucesso: C:\Users\Gabriel\OneDrive - I Care\I Care - 11. GEO\1. Polo Clima\1. Projetos Ativos\MAU1\data\raw\pr_Xavier_manaus_1961_2024.nc
Processando variável: RH
Arquivo salvo com sucesso: C:\Users\Gabriel\OneDrive - I Care\I Care - 11. GEO\1. Polo Clima\1. Projetos Ativos\MAU1\data\raw\RH_Xavier_manaus_1961_2024.nc
Processando variável: ET
Arquivo salvo com sucesso: C:\Users\Gabriel\OneDrive - I Care\I Care - 11. GEO\1. Polo Clima\1. Projetos Ativos\MAU1\data\raw\ET_Xavier_manaus_1961_2024.nc
Processando variável: Tmin
Arquivo salvo com sucesso: C:\Users\Gabriel\OneDrive - I Care\I Care - 11. GEO\1. Polo Clima\1. Projetos Ativos\MAU1\data\raw\Tmin_Xavier_manaus_1961_2024.nc
Processamento concluído para todas as variáveis!


In [ ]:
rescale_params = {
    "Rs": {"scale": 0.15708661, "offset": -0.057087},
    "u2": {"scale": 0.05905512, "offset": -0.059055},
    "Tmax": {"scale": 0.00106815, "offset": 15.0},
    "Tmin": {"scale": 0.00106815, "offset": 15.0},
    "RH": {"scale": 0.39370079, "offset": -0.393701},
    "pr": {"scale": 0.00686666, "offset": 225.0},
    "ET": {"scale": 0.05118110, "offset": 0.0},
}

variaveis_interesse = ["Tmax", "pr", "RH", "ET", "Tmin"]

# 3. Carregar coleção filtrada
var_name = "Tmax"
scale = rescale_params[var_name]["scale"]
offset = rescale_params[var_name]["offset"]
start = "1961-01-01"
end = "2024-03-20"
col = (
    ee.ImageCollection("projects/ee-alexandrexavier/assets/BR-DWGD")
    .filterBounds(area)
    .filterDate(start, end)
    .map(scaleBand)
    .select(var_name)
)

ds = xr.open_dataset(
    col,
    engine="ee",
    geometry=area,
    scale=0.1,
    projection=col.first().select(0).projection(),
    fast_time_slicing=True,
    chunks={"index": 24},
).rename({"lon": "x", "lat": "y"})

ds = ds.transpose("time", "y", "x").rio.write_crs("EPSG:5880")

output_folder = r"C:\Users\Gabriel\OneDrive - I Care\I Care - 11. GEO\1. Polo Clima\1. Projetos Ativos\MAU1\data\raw"
output_path = os.path.join(output_folder, "pr_Xavier_manaus_1961_2024.nc")
encoding = {var_name: {"zlib": True}}
# exportar netcdf
ds.to_netcdf(
    os.path.join(output_folder, f"{var_name}_Xavier_manaus_1961_2024.nc"),
    encoding=encoding,
    format="NETCDF4",
)